# 03. Stage 1B: Collaborative Filtering (Implicit ALS)

Notebook này xây dựng tầng lọc thô dựa trên hành vi tương tác cộng tác của người dùng (Collaborative Filtering) thông qua dữ liệu ngầm định (Implicit Feedback).

---

### Phân tích Quyết định Thiết kế:
*   **Tại sao chọn Implicit ALS (iALS)?**
    *   Trong thực tế, dữ liệu tương tác của người dùng chủ yếu là ngầm định (click, xem phim, tìm kiếm) chứ không có ratings tường minh. Nếu ta áp dụng SVD truyền thống, ta buộc phải coi phim chưa xem là nhãn âm (0), điều này sai vì có thể họ chưa biết phim đó. **Implicit ALS** giải quyết triệt để bằng cách xem tất cả tương tác là thước đo độ tin cậy (confidence matrix) kết hợp với các latent factors để mô hình hóa sở thích ẩn.
*   **Tại sao không chọn BPR-MF làm giải thuật chính ở Retrieval?**
    *   BPR-MF tối ưu hóa ranking cặp (pairwise) rất tốt cho danh sách ngắn, nhưng iALS hoạt động dựa trên toàn bộ ma trận (pointwise matrix factorization với confidence weights), giúp tận dụng tối đa tần suất và loại sự kiện khác nhau (click vs watch_complete) dễ dàng hơn. BPR chỉ nhận nhãn nhị phân (1/0).
*   **Tại sao không chọn Collaborative Filtering dựa trên lân cận (KNN)?**
    *   KNN yêu cầu lưu trữ và tính toán ma trận tương tương giữa tất cả các cặp User hoặc Item ($O(N^2)$ hoặc $O(M^2)$). Điều này gây tốn bộ nhớ nghiêm trọng và không thể mở rộng (scale) khi hệ thống đạt hàng chục nghìn người dùng.


In [1]:
import os
import pandas as pd
import numpy as np
import scipy.sparse as sp
import implicit
import pickle

# Load dữ liệu tương tác ngầm định
clicks_df = pd.read_csv(os.path.join("..", "..", "data", "simulator", "sim_click_events.csv"))
movies_df = pd.read_csv(os.path.join("..", "..", "data", "crawler", "movies_crawled.csv"))

# Encode userId và movieId sang dạng index liên tục
unique_users = clicks_df['userId'].unique()
unique_movies = movies_df['movieId'].unique()

user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
movie_to_idx = {mid: i for i, mid in enumerate(unique_movies)}
idx_to_movie = {i: mid for mid, i in movie_to_idx.items()}

# Lưu dict map để dùng lại
os.makedirs("processed_data", exist_ok=True)
with open("processed_data/id_mappings.pkl", "wb") as f:
    pickle.dump((user_to_idx, movie_to_idx, idx_to_movie), f)


In [2]:
# 1. Xây dựng ma trận tương tác có trọng số từ Behavior Funnel
event_weights = {
    'click': 1.0,
    'detail_view': 2.0,
    'watch_start': 3.0,
    'watch_complete': 5.0
}

clicks_df['weight'] = clicks_df['event_type'].map(event_weights)
user_movie_weights = clicks_df.groupby(['userId', 'movieId'])['weight'].sum().reset_index()

user_movie_weights = user_movie_weights[user_movie_weights['movieId'].isin(movie_to_idx.keys())]

user_indices = user_movie_weights['userId'].map(user_to_idx).values
item_indices = user_movie_weights['movieId'].map(movie_to_idx).values
weights = user_movie_weights['weight'].values

num_users = len(user_to_idx)
num_items = len(movie_to_idx)

user_item_matrix = sp.csr_matrix((weights, (user_indices, item_indices)), shape=(num_users, num_items))
print(f"Sparse matrix density: {100 * user_item_matrix.nnz / (num_users * num_items):.4f}%")


Sparse matrix density: 1.1260%


In [3]:
# 2. Huấn luyện Implicit ALS model
alpha = 40
sparse_user_item = (user_item_matrix * alpha).astype('double')

model = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.1,
    iterations=20,
    random_state=42
)

model.fit(sparse_user_item)

with open("models/als_model.pkl", "wb") as f:
    pickle.dump(model, f)
    
with open("processed_data/user_item_matrix.pkl", "wb") as f:
    pickle.dump(user_item_matrix, f)


C:\Users\Lenovo\anaconda3\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: Intel MKL BLAS is configured to use 14 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

In [4]:
# 3. Hàm đề xuất Collaborative Filtering candidates
def get_als_candidates(user_id, top_n=100):
    u_idx = user_to_idx.get(user_id, None)
    if u_idx is None:
        return movies_df.sort_values(by='popularity', ascending=False)['movieId'].head(top_n).tolist()
        
    ids, scores = model.recommend(u_idx, user_item_matrix[u_idx], N=top_n, filter_already_liked_items=True)
    recommended_movie_ids = [idx_to_movie[i] for i in ids]
    return recommended_movie_ids

# Thử nghiệm đề xuất
test_user = unique_users[0]
candidates = get_als_candidates(test_user, top_n=5)
print(f"Gợi ý ALS cho User {test_user}:", movies_df[movies_df['movieId'].isin(candidates)]['title'].tolist())


Gợi ý ALS cho User 1: ['My Fault: London', 'Scream VI', 'Thieves Highway', 'Expend4bles', 'Wonder Woman 1984']
